# Study-level Meta-regression with PyMetaAnalysis

This notebook uses **synthetic** study-level effects and moderators. It demonstrates the public Meta-regression workflow and does not represent a clinical question or justify a causal interpretation.

In [ ]:
%matplotlib inline

import pandas as pd

import meta_analyze as ma

## Prepare study-level effects and moderators

The analysis accepts standard errors directly and records their conversion to sampling variances. Because `study=` is omitted, the DataFrame index becomes the study label.

In [ ]:
studies = pd.DataFrame(
    {
        "effect": [
            0.350,
            0.745,
            0.335,
            0.225,
            1.005,
            0.230,
            1.470,
            0.165,
            1.020,
            0.745,
            0.385,
            0.960,
        ],
        "standard_error": [
            0.200,
            0.245,
            0.224,
            0.283,
            0.212,
            0.265,
            0.235,
            0.300,
            0.224,
            0.274,
            0.255,
            0.245,
        ],
        "dose": [0.5, 1.8, 1.1, 0.7, 2.4, 1.4, 2.0, 0.9, 2.7, 1.6, 0.4, 2.2],
        "region": ["North", "South", "East"] * 4,
    },
    index=pd.Index([f"Study {number}" for number in range(1, 13)], name="study"),
)
studies

## Fit a one-moderator mixed-effects model

REML estimates residual heterogeneity. Hartung-Knapp inference uses a t distribution with residual degrees of freedom for coefficients and prediction intervals.

In [ ]:
dose_model = ma.meta_regression(
    studies,
    effect="effect",
    standard_error="standard_error",
    moderators=["dose"],
    model="mixed",
    tau2_method="REML",
    inference_method="hartung_knapp",
)
print(dose_model.summary())

## Inspect coefficients and row diagnostics

There is no single pooled effect in Meta-regression. Inspect the intercept and slope together with their uncertainty, then retain row-level fitted values, residuals, precision weights, and leverage for audit.

In [ ]:
display(dose_model.coefficients)
display(
    dose_model.study_results[
        [
            "study",
            "effect",
            "dose",
            "fitted_value",
            "residual",
            "normalized_precision_weight",
            "leverage",
        ]
    ]
)

## Predict mean and true effects

For a mixed-effects model, `predict()` returns a confidence interval for the mean relationship and a prediction interval for a true effect at each supplied moderator value.

In [ ]:
new_doses = pd.DataFrame(
    {"dose": [0.75, 1.50, 2.25]},
    index=["low", "middle", "high"],
)
dose_model.predict(new_doses)

## Draw the weighted bubble plot

Marker area reflects normalized precision weight. The shaded band is the fitted mean confidence interval; enable the prediction interval when it helps answer the reporting question.

In [ ]:
ax = dose_model.bubble(
    moderator_label="Dose",
    effect_label="Synthetic effect",
    show_prediction_interval=True,
)
ax.figure.set_size_inches(8, 5)
ax

## Add an explicitly encoded categorical moderator

Categorical levels are never inferred silently. The first declared level is the treatment-coding reference, and `test_moderator()` jointly tests the non-reference terms.

In [ ]:
adjusted_model = ma.meta_regression(
    studies,
    effect="effect",
    standard_error="standard_error",
    moderators=["dose", "region"],
    categorical={"region": ["North", "South", "East"]},
    model="mixed",
    tau2_method="REML",
    inference_method="hartung_knapp",
)
display(adjusted_model.coefficients)
adjusted_model.test_moderator("region")

In [ ]:
prediction_rows = pd.DataFrame(
    {
        "dose": [1.0, 1.5, 2.0],
        "region": ["North", "South", "East"],
    }
)
adjusted_model.predict(prediction_rows)

## Preserve methods and provenance

Generated text and structured reports are starting points for review. They record the resolved design, method choices, row decisions, diagnostics, and package version.

In [ ]:
print(adjusted_model.method_details())

report = adjusted_model.report(include_studies=True)
payload = report.to_dict()
print(payload["schema_version"], payload["report_type"])
sorted(payload)

## Interpretation boundary

Moderator coefficients describe study-level associations. They do not establish individual-level relationships or causality, and a small number of studies provides limited power for detecting moderator effects. Prespecify important moderators, avoid overfitting, retain the exact package version and report, and independently check consequential analyses.